In [1]:
import csv

In [2]:
SV_FILE = '../../../hybrid_diversity/5_snp_analyses/5_manual_SV_annotation/svlist_v2.txt'
LOG2_PATH = '../../../hybrid_diversity/4_read_mapping/3_deduplicated/2_nuc_and_chloroplast/'
ALLELIC_PATH = '../../../hybrid_diversity/5_snp_analyses/3_aims/3_vcf/allelic_depth_per_sample/'
OUTFILE = '../../../hybrid_diversity/5_snp_analyses/5_manual_SV_annotation/svlist_v2_statistics.txt'

# Set to None to disable skipping, or provide a path to a BED file.
# BED format: chrom, start, end (0-based, half-open intervals), optional extra columns ignored.
SKIP_REGIONS_FILE = 'skip_regions.bed'

In [3]:
def load_skip_regions(filepath):
    """
    Load a BED file of regions to skip.
    Returns a dict mapping chrom -> list of (start, end) tuples (0-based, half-open).
    Returns an empty dict if filepath is None.
    """
    skip = {}
    if filepath is None:
        return skip
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            chrom, start, end = parts[0], int(parts[1]), int(parts[2])
            skip.setdefault(chrom, []).append((start, end))
    return skip


def overlaps_skip_region(chrom, pos_start, pos_end, skip_regions):
    """
    Return True if the interval [pos_start, pos_end) overlaps any region
    in skip_regions for the given chrom.
    pos_start and pos_end are in base-pair coordinates (0-based, half-open).
    """
    for (sk_start, sk_end) in skip_regions.get(chrom, []):
        if pos_end > sk_start and pos_start < sk_end:
            return True
    return False


# Load skip regions once at the top level so both query functions can use them
SKIP_REGIONS = load_skip_regions(SKIP_REGIONS_FILE)
print(f"Loaded skip regions for {len(SKIP_REGIONS)} chromosome(s): "
      f"{sum(len(v) for v in SKIP_REGIONS.values())} total intervals")

Loaded skip regions for 3 chromosome(s): 3 total intervals


In [4]:
def query_log2ratio(filepath, scaffold, start, stop):
    """
    Return average log2ratio for windows that overlap [start, stop]
    and do NOT overlap any region in SKIP_REGIONS.
    The log2ratio file has columns: chr, start, end, log2ratio.
    A window is included if it overlaps the query region.
    """

    start = start * 1000000
    stop = stop * 1000000
    
    values = []
    try:
        with open(filepath, newline="") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                if row["chr"] != scaffold:
                    continue
                win_start = int(row["start"])
                win_end = int(row["end"])
                
                # Overlap check: window overlaps [start, stop]
                if win_end > start and win_start <= stop:
                    # Skip if this window overlaps a skip region
                    #if overlaps_skip_region(scaffold, win_start, win_end, SKIP_REGIONS):
                    #    print('skipping', row["chr"], row["start"], row["end"])
                    #    continue
                        
                    values.append(float(row["log2ratio"]))
    
    except FileNotFoundError:
        sys.exit(f"ERROR: log2ratio file not found: {filepath}")

    if not values:
        return None, 0
    return sum(values) / len(values), len(values)


In [5]:
def query_allelic_depth(filepath, scaffold, start, stop):
    """
    Return average A1 depth, A2 depth, A1 frac, and A2 frac for SNPs
    that fall within [start, stop] and do NOT overlap any region in SKIP_REGIONS.
    """
    start = start * 1000000
    stop = stop * 1000000
    
    a1_values = []
    a2_values = []
    a1_frac_values = []
    a2_frac_values = []
    
    try:
        with open(filepath, newline="") as f:
            reader = csv.DictReader(f, delimiter="\t")
            for row in reader:
                if row["CHROM"] != scaffold:
                    continue
                pos = int(row["POS"])
                if start <= pos <= stop:
                    # Skip if this SNP position overlaps a skip region
                    # SNP is a single base: treat as [pos, pos+1) for BED-style overlap
                    #if overlaps_skip_region(scaffold, pos, pos + 1, SKIP_REGIONS):
                    #    print('skipping', row["CHROM"], row["POS"])
                    #    continue
                        
                    a1_values.append(float(row["A1_depth"]))
                    a2_values.append(float(row["A2_depth"]))

                    if row["A1_frac"] != '.' and row["A2_frac"] != '.':
                        a1_frac_values.append(float(row["A1_frac"]))
                        a2_frac_values.append(float(row["A2_frac"]))
    
    except FileNotFoundError:
        sys.exit(f"ERROR: allelic depth file not found: {filepath}")

    if not a1_values:
        return None, None, None, None, 0

    avg_a1 = sum(a1_values) / len(a1_values)
    avg_a2 = sum(a2_values) / len(a2_values)
    avg_a1_frac = sum(a1_frac_values) / len(a1_frac_values)
    avg_a2_frac = sum(a2_frac_values) / len(a2_frac_values)
    
    return avg_a1, avg_a2, avg_a1_frac, avg_a2_frac, len(a1_frac_values)


In [6]:
with open(SV_FILE, 'r') as fi, open(OUTFILE, 'w') as fo:
    fo.write(f"#strain\tscaffold\tstart\tstop\tavg_log2_coverage\tn_coverage_windows\tavg_a1_allele_depth\tavg_a2_allele_depth\tavg_a1_allele_fraction\tavg_a2_allele_fraction\tn_snps\n")

    for line in fi:
        if line[0] == '#':
            continue
            
        strain, scaf, start, stop = line.rstrip().split('\t')
        scaf = 'scaf' + scaf
        print(strain, scaf, start, stop)

        start = float(start)
        stop = float(stop)

        log2_file = LOG2_PATH + strain + ".log2ratio.tsv"
        allelic_file = ALLELIC_PATH + strain + ".allelic_depth.tsv"

        avg_a1, avg_a2, avg_a1_frac, avg_a2_frac, n_snps = query_allelic_depth(allelic_file, scaf, start, stop)
        avg_log2, n_windows = query_log2ratio(log2_file, scaf, start, stop)

        fo.write(f"{strain}\t{scaf}\t{start}\t{stop}\t{avg_log2}\t{n_windows}\t{avg_a1}\t{avg_a2}\t{avg_a1_frac}\t{avg_a2_frac}\t{n_snps}\n")
        

12A1-SC2 scaf8 2.05 2.9
12A1-SC3 scaf8 0 0.7
UTEX2797-SC2 scaf9 0 0.3
UTEX2797-SC3 scaf9 0 0.3
UTEX2797-SC1 scaf9 0 3.65
12A1-SC1 scaf8 0 3.65
UTEX2797-SC2 scaf22 0 1.85
UTEX2797-SC3 scaf22 0 1.85
ARC82 scaf4 2.9 3.3
